# CAVAS — Deep Learning Model Evaluation
### IDS Intrusion Detection: TabNet (Tabular) + TFT (Time Series)

**Dual prediction targets:**
- `label_generic` → Binary classification (benign / malicious)
- `Label` → Multiclass classification (benign / attack type)

**Models:**
- **TabNet** — sequential attention, native feature importance
- **Temporal Fusion Transformer (TFT)** — variable selection networks + temporal attention

**Hyperparameter tuning:** Optuna (TPE sampler)

---

### Notebook Workflow (5 steps)
1. **HPO (0.1% dataset):** Optuna hyperparameter search for TabNet & TFT — save all trial models + confusion matrices
2. **Best models & Feature Importance:** Extract best trial per model, plot feature importance
3. **Top features intersection:** Top-10 features from each model → intersection (~8–10 features)
4. **10% dataset (reduced features):** Stratified 10% sample, keep only important features
5. **Baseline training:** Retrain both models on the 10% reduced-feature dataset with optimized hyperparams

In [26]:
LOCAL_RUN = True
RANDOM_SEED = 42
RUNNINNG_ON_LINIX = True
TRIALS_ALREADY_EXECUTED = True
MIN_SAMPLES_PER_CLASS = 5

PERCENTAGE_TO_USE = 0.1  
WINDOW_SIZE = 50   # finestra temporale per CNN-LSTM
STEP_SIZE   = 10   # overlap tra finestre

## 0. Configuration & Setup

In [27]:
!pip install -q pytorch-tabnet pytorch-forecasting pytorch-lightning optuna optuna-integration scikit-learn pandas pyarrow


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [28]:
import pucktrick
from pucktrick import Engine
from pucktrick import PuckTrick
import os, subprocess, warnings, json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import math

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer

# Sklearn
from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.metrics          import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, matthews_corrcoef
)

# TabNet — multi-task classifier (predicts both targets simultaneously)
from pytorch_tabnet.multitask import TabNetMultiTaskClassifier

# TFT — use lightning.pytorch (NOT pytorch_lightning) to match pytorch_forecasting
import torch
import lightning.pytorch as pl

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Optuna
import optuna
from optuna.integration import PyTorchLightningPruningCallback
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
pl.seed_everything(RANDOM_SEED)
print("All imports OK")

Seed set to 42


All imports OK


In [29]:
# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION — change only here
# ─────────────────────────────────────────────────────────────────────
PATH_IMG = "images"

if LOCAL_RUN:
    PATH = "DATASETS"
    TABNET_MAX_EPOCHS  = 20
    CNN_LSTM_MAX_EPOCHS= 50
else:
    PATH = "file:///home/PuckTrickadmin/DATASETS"
    TABNET_MAX_EPOCHS  = 150
    CNN_LSTM_MAX_EPOCHS= 100

RANDOM_SEED  = 42
# Split temporale: non servono più TEST_SIZE / VAL_SIZE
# La suddivisione è deterministica: ogni 3 pacchetti → 2 train, 1 temp
# Da temp → alternato val / test (≈66.7% train, 16.7% val, 16.7% test)

os.makedirs(PATH_IMG, exist_ok=True)
os.makedirs("models",  exist_ok=True)

In [30]:
# ── Spark session (LOCAL / SERVER) ──────────────────────────────────
if LOCAL_RUN:
    if (RUNNINNG_ON_LINIX):
        java_home = os.environ.get('JAVA_HOME', '')
        if not java_home:
            try:
                java_path = subprocess.check_output(['which', 'java'], text=True).strip()
                os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(os.path.realpath(java_path)))
            except subprocess.CalledProcessError:
                print("⚠️  Java not found — run: sudo apt install default-jdk")

        os.environ['PYSPARK_PYTHON']        = 'python3'
        os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

        spark = SparkSession.builder \
            .appName("CAVAS_Models") \
            .master("local[*]") \
            .config("spark.driver.memory",          "30g") \
            .config("spark.driver.host",            "localhost") \
            .config("spark.ui.showConsoleProgress", "false") \
            .getOrCreate()
        
    else:
        # Forza JAVA_HOME al JRE corretto
        os.environ['JAVA_HOME'] = r"C:\Program Files\Java\jre-1.8"

        # Hadoop winutils per Windows
        os.environ['HADOOP_HOME'] = r"C:\hadoop"
        os.environ['PATH'] = os.environ.get('PATH', '') + r';C:\hadoop\bin'

        # Su Windows l'eseguibile è 'python', non 'python3'
        py = 'python' if os.name == 'nt' else 'python3'
        os.environ['PYSPARK_PYTHON']        = py
        os.environ['PYSPARK_DRIVER_PYTHON'] = py

        spark = SparkSession.builder \
            .appName("CAVAS_Models") \
            .master("local[*]") \
            .config("spark.driver.memory",          "24g") \
            .config("spark.driver.host",            "localhost") \
            .config("spark.ui.showConsoleProgress", "false") \
            .getOrCreate()
        
else:
    MASTER_URL  = "spark://10.0.1.8:7077"
    DRIVER_HOST = "10.0.1.8"

    spark = SparkSession.builder \
        .appName("CAVAS_Models") \
        .master(MASTER_URL) \
        .config("spark.submit.deployMode",      "client") \
        .config("spark.executor.instances",     "4") \
        .config("spark.executor.cores",         "4") \
        .config("spark.executor.memory",        "13g") \
        .config("spark.driver.memory",          "8g") \
        .config("spark.driver.host",            DRIVER_HOST) \
        .config("spark.driver.bindAddress",     DRIVER_HOST) \
        .config("spark.sql.shuffle.partitions", "32") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

print(f"✅  Spark {spark.version} ready")

✅  Spark 3.5.0 ready


26/04/29 07:30:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 0.1. Data Loading: Stratified Sample + Timestamp Cleanup

In [31]:
# Feature types from your analysis
CATEGORICAL_FEATURES = ['Fwd Seg Size Min', 'Protocol']
BINARY_FEATURES      = ['FIN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'Fwd URG Flag', 'Fwd PSH Flag']

In [32]:
def label_encoding_spark(sdf):
    """Add label_generic_enc (0/1) and Label_enc (integer) to the Spark DataFrame.
    Returns (sdf_encoded, label_classes) where label_classes maps index → Label name.
    """
    # ── Binary: label_generic is already 0/1 → cast to int ──────────
    sdf = sdf.withColumn('label_generic_enc', col('label_generic').cast('int'))

    # ── Multiclass: Label → integer index via StringIndexer ──────────
    indexer = StringIndexer(inputCol='Label', outputCol='Label_enc', handleInvalid='keep')
    model = indexer.fit(sdf)
    sdf = model.transform(sdf)
    sdf = sdf.withColumn('Label_enc', col('Label_enc').cast('int'))

    # Store ordered label list: index 0 → labels[0], etc.
    label_classes = list(model.labels)

    n_binary = sdf.select('label_generic_enc').distinct().count()
    print(f"✅  label_generic_enc: {n_binary} classes | Label_enc: {len(label_classes)} classes")
    print(f"    Label mapping: { {i: l for i, l in enumerate(label_classes)} }")
    return sdf, label_classes

In [33]:
def preprocess_to_pandas(sdf, continuous_features, categorical_features, binary_features):
    """
    Convert Spark → Pandas and clean dtypes.
    
    - Continuous: cast to float64 (scaling done AFTER train/test split to avoid leakage)
    - Categorical: integer-encoded via LabelEncoder (TabNet/TFT handle them natively)
    - Binary: cast to int
    
    ⚠ NO one-hot encoding:
      • TabNet uses cat_idxs / cat_dims natively
      • TFT uses time_varying_known_categoricals
      • Feature importance stays traceable to the original feature name
    """
    print("⏳  Converting Spark → Pandas ...")
    pdf = sdf.toPandas()
    print(f"📊  Shape: {pdf.shape}")

    available = set(pdf.columns)
    cont_cols = [c for c in continuous_features if c in available]
    cat_cols  = [c for c in categorical_features if c in available]
    bin_cols  = [c for c in binary_features if c in available]

    # ── Continuous → float64 ──────────────────────────────────────────
    for c in cont_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce')
    pdf[cont_cols] = pdf[cont_cols].fillna(0.0)

    # ── Categorical → integer codes ──────────────────────────────────
    cat_encoders = {}
    cat_dims = {}
    for c in cat_cols:
        le = LabelEncoder()
        pdf[c] = le.fit_transform(pdf[c].astype(str))
        cat_encoders[c] = le
        cat_dims[c] = len(le.classes_)

    # ── Binary → int ─────────────────────────────────────────────────
    for c in bin_cols:
        pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0).astype(int)

    print(f"✅  Preprocessed: {len(cont_cols)} continuous | {len(cat_cols)} categorical | {len(bin_cols)} binary")
    return pdf, cat_encoders, cat_dims

## Step 1: Import Models functions

Both models are tuned via Optuna on a 0.1% stratified sample.  
Each trial's model is stored in memory along with confusion matrices for both targets.

### Step 1.0. Utility functions

In [34]:
# ── Utility function for metrics ─────────────────────────────────────
def print_metrics(y_true, y_pred, y_proba, task_name, class_names=None, verbose=1):
    is_binary = (len(np.unique(y_true)) == 2)
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='binary' if is_binary else 'macro')
    try:
        auc = roc_auc_score(
            y_true,
            y_proba[:, 1] if is_binary else y_proba,
            multi_class='ovr' if not is_binary else 'raise'
        )
    except Exception:
        auc = float('nan')

    if verbose != 0:
        print(f"\n{'='*55}")
        print(f"  {task_name}")
        print(f"{'='*55}")
        print(f"  Accuracy : {acc:.4f}  |  F1: {f1:.4f}  |  MCC: {mcc:.4f}  |  AUC: {auc:.4f}")
    present_labels = sorted(np.unique(np.concatenate([np.unique(y_true), np.unique(y_pred)])))
    if class_names is not None:
        target_names_filtered = [class_names[i] for i in present_labels if i < len(class_names)]
    else:
        target_names_filtered = None
    if verbose != 0:
        print(classification_report(y_true, y_pred, labels=present_labels,
                                    target_names=target_names_filtered))
    return dict(task=task_name, accuracy=acc, f1=f1, mcc=mcc, auc=auc)

In [35]:
# ── Utility: show stored model report ─────────────────────────────────
def show_model_report(model_name, artifacts_dict=None):
    """
    Display confusion matrices + key metrics for a previously-run trial.
    
    Parameters
    ----------
    model_name : str or int
        The key used in tabnet_trial_artifacts / tft_trial_artifacts,
        e.g. trial number (int) or 'Baseline' / 'TFT_Baseline' (str).
    artifacts_dict : dict, optional
        If provided, look up model_name in this dict directly.
        Otherwise tries tabnet_trial_artifacts first, then tft_trial_artifacts.
    """
    # ── Locate the artifact ───────────────────────────────────────────
    art = None
    if artifacts_dict is not None:
        art = artifacts_dict.get(model_name)
    else:
        art = tabnet_trial_artifacts.get(model_name) or cnn_lstm_trial_artifacts.get(model_name)

    if art is None:
        # Try loading from JSON on disk
        for prefix in ['tabnet_trial_', 'cnn_lstm_trial_']:
            path = f'models/{prefix}{model_name}_artifacts.json'
            if os.path.exists(path):
                with open(path) as f:
                    art = json.load(f)
                break
    if art is None:
        print(f"❌  No artifacts found for '{model_name}'")
        return

    model_type = art.get('model_type', 'Unknown')
    label      = art.get('label', str(model_name))

    print(f"\n{'='*60}")
    print(f"  📊  Report: {model_type} — {label}")
    print(f"{'='*60}")

    # ── Scalar metrics ────────────────────────────────────────────────
    for task_key, task_label in [('metrics_binary', 'Binary Task'),
                                  ('metrics_multiclass', 'Multiclass Task')]:
        m = art.get(task_key)
        if m is None:
            print(f"\n  ⚠️  {task_label}: metrics not available")
            continue
        print(f"\n  {task_label}:")
        print(f"    Accuracy : {m['accuracy']:.4f}")
        print(f"    F1-score : {m['f1']:.4f}")
        print(f"    MCC      : {m['mcc']:.4f}")
        print(f"    AUC      : {m['auc']:.4f}")

    # ── Extra scalars (model-specific) ────────────────────────────────
    if 'mean_mcc' in art:
        print(f"\n  Mean MCC (binary+multi): {art['mean_mcc']:.4f}")
    if 'val_loss' in art:
        print(f"  Val loss: {art['val_loss']:.4f}")

    # ── Confusion matrices ────────────────────────────────────────────
    cm_bin = art.get('cm_bin')
    cm_mul = art.get('cm_mul')

    if cm_bin is None and cm_mul is None:
        print("\n  ⚠️  No confusion matrices available for this trial")
        return

    # Convert from list-of-lists (JSON) back to ndarray if needed
    if cm_bin is not None and not isinstance(cm_bin, np.ndarray):
        cm_bin = np.array(cm_bin)
    if cm_mul is not None and not isinstance(cm_mul, np.ndarray):
        cm_mul = np.array(cm_mul)

    n_plots = (cm_bin is not None) + (cm_mul is not None)
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    idx = 0
    if cm_bin is not None:
        sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
        axes[idx].set_title(f'{label} — Binary CM')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')
        idx += 1

    if cm_mul is not None:
        sns.heatmap(cm_mul, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
        axes[idx].set_title(f'{label} — Multiclass CM')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')

    plt.tight_layout()
    plt.show()
    print(f"\n  Params: {json.dumps(art.get('params', {}), indent=4)}")
    
    # ── Feature importance ────────────────────────────────────────────
    fi = art.get('feature_importance')
    if fi:
        fi_sorted = dict(sorted(fi.items(), key=lambda x: x[1], reverse=True))
        top_n = dict(list(fi_sorted.items())[:20])  # top 20

        print(f"\n  Top feature importances (top {len(top_n)}):")
        fig_fi, ax_fi = plt.subplots(figsize=(8, max(3, len(top_n) * 0.35)))
        ax_fi.barh(list(top_n.keys())[::-1], list(top_n.values())[::-1], color='steelblue')
        ax_fi.set_xlabel('Importance')
        ax_fi.set_title(f'{label} — Feature Importance')
        plt.tight_layout()
        plt.show()
    else:
        print("\n  ⚠️  Feature importance not available for this trial")

In [36]:
def json_serializer(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return None if np.isnan(x) else x.item()
    if isinstance(x, float) and np.isnan(x):
        return None  # NaN → null in JSON
    return str(x)

### Step 1a — TabNet Hyperparameter Tuning (Optuna)

**Key TabNet hyperparameters:**
| Param | Meaning |
|---|---|
| `N_a` / `N_d` | Width of attention + decision steps (usually equal) |
| `N_steps` | Number of sequential attention steps |
| `gamma` | Sparsity regularisation coefficient |
| `lambda_sparse` | Feature sparsity penalty |
| `lr` | Learning rate |

Each trial's model, confusion matrices (binary + multiclass), and metrics are saved in memory.

In [37]:
#tabnet_trial_artifacts = {}

In [38]:
# ── Global storage for TabNet trial artifacts ─────────────────────────
def tabnet_multitask_objective(X_train, X_val,
                                y_tr_bin, y_tr_mul,
                                y_val_bin, y_val_mul,
                                N_a, N_steps, gamma, lambda_s, lr, batch_sz, mask_type,
                                verbose=0, trial=None, label_model=None,
                                feature_names=None,
                                corr_matrix=None):
    """
    Stores model + confusion matrices in memory for each trial.
    Returns mean MCC across both tasks on validation set.
    If a saved model with the same label_model exists on disk,
    reloads it, skips training, recomputes all metrics, and re-saves the JSON.
    Feature importance is loaded from the existing JSON when reloading.
    """
    FINAL_LABEL = label_model if label_model is not None else trial.number

    model_path     = f'experiments/tabnet_trial_{FINAL_LABEL}'
    artifacts_path = f'experiments/tabnet_trial_{FINAL_LABEL}_artifacts.json'

    reloaded = False
    old_feature_importance = None

    # ── Check if model already exists → reload & skip training ────────
    if os.path.exists(model_path + '.zip'):
        print(f"♻️  Found existing TabNet model for {FINAL_LABEL}, reloading...")
        reloaded = True
        # Load feature importance from existing JSON (skip recalculation)
        if os.path.exists(artifacts_path):
            with open(artifacts_path) as f:
                old_art = json.load(f)
            old_feature_importance = old_art.get('feature_importance')
        clf = TabNetMultiTaskClassifier()
        clf.load_model(model_path + '.zip')
    else:
        # ── Train from scratch ────────────────────────────────────────
        clf = TabNetMultiTaskClassifier(
            n_d=N_a, n_a=N_a,
            n_steps=N_steps,
            gamma=gamma,
            lambda_sparse=lambda_s,
            cat_idxs=CAT_IDXS if CAT_IDXS else [],
            cat_dims=CAT_DIMS if CAT_DIMS else [],
            cat_emb_dim=1,
            optimizer_params=dict(lr=lr),
            mask_type=mask_type,
            verbose=verbose,
            seed=RANDOM_SEED,
        )

        y_train_mt = np.column_stack([y_tr_bin, y_tr_mul])
        y_val_mt   = np.column_stack([y_val_bin, y_val_mul])

        clf.fit(
            X_train,
            y_train_mt,
            eval_set      = [(X_val, y_val_mt)],
            eval_metric   = ['accuracy'],
            max_epochs    = TABNET_MAX_EPOCHS,
            patience      = 4,
            batch_size    = batch_sz,
            virtual_batch_size = max(batch_sz // 4, 64),
            drop_last     = False,
        )

        clf.save_model(model_path)
        print(f"✅  Saved TabNet model for trial {FINAL_LABEL}")

    # ── Evaluation (always runs — recomputes all metrics) ─────────────
    raw_preds = clf.predict(X_val)
    pred_bin = np.asarray(raw_preds[0]).astype(int)
    pred_mul = np.asarray(raw_preds[1]).astype(int)
    y_val_bin_int = np.asarray(y_val_bin).astype(int)
    y_val_mul_int = np.asarray(y_val_mul).astype(int)

    mcc_bin = matthews_corrcoef(y_val_bin_int, pred_bin)
    mcc_mul = matthews_corrcoef(y_val_mul_int, pred_mul)

    cm_bin = confusion_matrix(y_val_bin_int, pred_bin)
    cm_mul = confusion_matrix(y_val_mul_int, pred_mul)

    if verbose != 0:
        print(f"\nConfusion Matrix - Trial {FINAL_LABEL}:")
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Benign', 'Malicious'],
                    yticklabels=['Benign', 'Malicious'])
        axes[0].set_title(f'Trial {FINAL_LABEL} — Binary CM')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Actual')

        sns.heatmap(cm_mul, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                    xticklabels=label_classes[:cm_mul.shape[1]],
                    yticklabels=label_classes[:cm_mul.shape[0]])
        axes[1].set_title(f'Trial {FINAL_LABEL} — Multiclass CM')
        axes[1].set_xlabel('Predicted')
        axes[1].set_ylabel('Actual')
        plt.xticks(rotation=45, ha='right')

        plt.tight_layout()
        plt.savefig(f'{PATH_IMG}/tabnet_trial{FINAL_LABEL}_cm.png', dpi=150, bbox_inches='tight')
        plt.show()
        
    proba = clf.predict_proba(X_val)
    metrics_first_output = print_metrics(y_val_bin_int, pred_bin, proba[0],
                      f'Trial {FINAL_LABEL} - Binary Task',
                      class_names=['Benign', 'Malicious'], verbose=verbose)
    metrics_second_output = print_metrics(y_val_mul_int, pred_mul, proba[1],
                      f'Trial {FINAL_LABEL} - Multiclass Task',
                      class_names=label_classes, verbose=verbose)
    
    # ── Feature importance: reuse from old JSON if reloaded ───────────
    if reloaded and old_feature_importance is not None:
        feature_importance = old_feature_importance
    else:
        if feature_names is not None:
            feat_names = list(feature_names)
        elif hasattr(X_train, 'columns'):
            feat_names = list(X_train.columns)
        else:
            feat_names = [f'f{i}' for i in range(X_train.shape[1])]
        importance_scores = clf.feature_importances_
        feature_importance = dict(zip(feature_names, importance_scores.tolist()))

    # ── Store model and artifacts in memory ───────────────────────────
    mean_mcc = (mcc_bin + mcc_mul) / 2
    object_to_store = {
        'model': f'tabnet_trial_{FINAL_LABEL}',
        'model_type': 'TabNet',
        'label': str(FINAL_LABEL),
        'mcc_bin': mcc_bin,
        'mcc_mul': mcc_mul,
        'mean_mcc': mean_mcc,
        'cm_bin': cm_bin,
        'cm_mul': cm_mul,
        'feature_importance': feature_importance,
        'params': trial.params if trial is not None else {
            'N_a': N_a, 'N_steps': N_steps, 'gamma': gamma,
            'lambda_sparse': lambda_s, 'lr': lr, 'batch_size': batch_sz, 'mask_type': mask_type
        },
        'metrics_binary': metrics_first_output,
        'metrics_multiclass': metrics_second_output,
        'correlation_matrix': corr_matrix,
    }

    with open(artifacts_path, 'w') as f:
        json.dump(object_to_store, f, indent=4, default=json_serializer)

    print(f"Trial {FINAL_LABEL}: MCC_bin={mcc_bin:.4f}, MCC_mul={mcc_mul:.4f}, mean={mean_mcc:.4f}")
    return mean_mcc

### Step 1b — CNN-LSTM (Time Series)

**Strategy: Single Continuous Time Series**

The entire dataset represents a **single continuous network capture session** ordered by `Timestamp`.
Flows are sorted chronologically and indexed sequentially to form a unified time series.
This allows TFT to learn temporal patterns in attack campaigns (port scans, DDoS bursts, etc.)
by looking at the sequence of flows over time.

Each trial's model, confusion matrices (when computable), and metrics are saved in memory.

In [39]:
# ── Global storage for TFT trial artifacts ────────────────────────────
#cnn_lstm_trial_artifacts = {}

In [40]:
class CNNLSTMMultiTask(nn.Module):
    def __init__(self, n_features, n_timesteps, n_classes_bin, n_classes_mul,
                 nb_filters=64, kernel_size=3, lstm_units_1=64,
                 lstm_units_2=128, dropout=0.3):
        super().__init__()

        # CNN opera su (batch, n_features, n_timesteps)
        # ogni feature è un canale, i timesteps sono la dimensione spaziale
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=n_features, out_channels=nb_filters,
                      kernel_size=kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.BatchNorm1d(nb_filters),
        )

        self.lstm1 = nn.LSTM(input_size=nb_filters, hidden_size=lstm_units_1,
                             batch_first=True)
        self.lstm2 = nn.LSTM(input_size=lstm_units_1, hidden_size=lstm_units_2,
                             batch_first=True)
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Sequential(
            nn.Linear(lstm_units_2, lstm_units_2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.head_bin = nn.Linear(lstm_units_2, n_classes_bin)
        self.head_mul = nn.Linear(lstm_units_2, n_classes_mul)

    def forward(self, x):
        # x: (batch, n_timesteps, n_features)
        x = x.permute(0, 2, 1)        # → (batch, n_features, n_timesteps)
        x = self.cnn(x)                # → (batch, nb_filters, timesteps//2)
        x = x.permute(0, 2, 1)        # → (batch, timesteps//2, nb_filters)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x = x[:, -1, :]               # ultimo timestep
        x = self.dropout(x)
        x = self.fc(x)
        return self.head_bin(x), self.head_mul(x)

In [41]:
def cnn_lstm_multitask_objective(X_train, X_val,
                                  y_tr_bin, y_tr_mul,
                                  y_val_bin, y_val_mul,
                                  nb_filters, kernel_size,
                                  lstm_units_1, lstm_units_2,
                                  dropout, lr, batch_size,
                                  verbose=0, trial=None, label_model=None,
                                  feature_names=None,
                                  corr_matrix=None):
    """
    X_train / X_val: 3D arrays (n_samples, n_timesteps, n_features)
    If a saved model with the same label_model exists on disk,
    reloads it, skips training, recomputes all metrics, and re-saves the JSON.
    Feature importance is loaded from the existing JSON when reloading.
    """
    FINAL_LABEL = label_model if label_model is not None else trial.number
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    n_features  = X_train.shape[2]
    n_timesteps = X_train.shape[1]

    # ── DataLoader (needed for both training and evaluation) ──────────
    def make_loader(X, yb, ym, shuffle):
        ds = TensorDataset(
            torch.tensor(X, dtype=torch.float32),   # (N, T, F)
            torch.tensor(yb, dtype=torch.long),
            torch.tensor(ym, dtype=torch.long),
        )
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

    train_dl = make_loader(X_train, y_tr_bin, y_tr_mul, shuffle=False)
    val_dl   = make_loader(X_val,   y_val_bin, y_val_mul, shuffle=False)

    loss_bin = nn.CrossEntropyLoss()
    loss_mul = nn.CrossEntropyLoss()

    model_path     = f'experiments/cnn_lstm_trial_{FINAL_LABEL}.pt'
    artifacts_path = f'experiments/cnn_lstm_trial_{FINAL_LABEL}_artifacts.json'
    val_losses = None  # populated only when training from scratch

    reloaded = False
    old_feature_importance = None

    # ── Check if model already exists → reload & skip training ────────
    if os.path.exists(model_path):
        print(f"♻️  Found existing CNN-LSTM model for {FINAL_LABEL}, reloading...")
        reloaded = True
        # Load feature importance from existing JSON (skip recalculation)
        if os.path.exists(artifacts_path):
            with open(artifacts_path) as f:
                old_art = json.load(f)
            old_feature_importance = old_art.get('feature_importance')
        model = CNNLSTMMultiTask(
            n_features    = n_features,
            n_timesteps   = n_timesteps,
            n_classes_bin = 2,
            n_classes_mul = int(max(np.max(y_tr_mul), np.max(y_val_mul))) + 1,
            nb_filters    = nb_filters,
            kernel_size   = kernel_size,
            lstm_units_1  = lstm_units_1,
            lstm_units_2  = lstm_units_2,
            dropout       = dropout,
        ).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
    else:
        # ── Train from scratch ────────────────────────────────────────
        model = CNNLSTMMultiTask(
            n_features    = n_features,
            n_timesteps   = n_timesteps,
            n_classes_bin = 2,
            n_classes_mul = int(max(np.max(y_tr_mul), np.max(y_val_mul))) + 1,
            nb_filters    = nb_filters,
            kernel_size   = kernel_size,
            lstm_units_1  = lstm_units_1,
            lstm_units_2  = lstm_units_2,
            dropout       = dropout,
        ).to(device)

        optimizer  = optim.Adam(model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=3,
            min_lr=1e-8
        )

        # ── Training ──────────────────────────────────────────────────
        best_val_loss = float('inf')
        patience_counter = 0
        PATIENCE = 10
        val_losses = []

        for epoch in range(CNN_LSTM_MAX_EPOCHS):
            model.train()
            for X_b, yb_b, ym_b in train_dl:
                X_b, yb_b, ym_b = X_b.to(device), yb_b.to(device), ym_b.to(device)
                optimizer.zero_grad()
                out_bin, out_mul = model(X_b)
                loss = loss_bin(out_bin, yb_b) + loss_mul(out_mul, ym_b)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            # Validation
            model.eval()
            val_loss_epoch = 0.0
            with torch.no_grad():
                for X_b, yb_b, ym_b in val_dl:
                    X_b, yb_b, ym_b = X_b.to(device), yb_b.to(device), ym_b.to(device)
                    out_bin, out_mul = model(X_b)
                    val_loss_epoch += (loss_bin(out_bin, yb_b) + loss_mul(out_mul, ym_b)).item()

            val_loss_epoch /= len(val_dl)
            val_losses.append(val_loss_epoch)

            scheduler.step(val_loss_epoch)

            if verbose != 0:
                print(f"  Epoch {epoch+1:3d} | val_loss: {val_loss_epoch:.4f}")

            # Early stopping
            if val_loss_epoch < best_val_loss:
                best_val_loss = val_loss_epoch
                patience_counter = 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if verbose != 0:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

            # Optuna pruning
            if trial is not None:
                trial.report(val_loss_epoch, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

        # Ripristina best weights
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), model_path)
        print(f"✅  Saved CNN-LSTM model for trial {FINAL_LABEL}")

    # ── Evaluation (always runs — recomputes all metrics) ─────────────
    model.eval()
    all_pred_bin, all_pred_mul = [], []
    all_prob_bin, all_prob_mul = [], []
    all_true_bin, all_true_mul = [], []
    eval_loss_total = 0.0

    with torch.no_grad():
        for X_b, yb_b, ym_b in val_dl:
            X_b, yb_b, ym_b = X_b.to(device), yb_b.to(device), ym_b.to(device)
            out_bin, out_mul = model(X_b)
            eval_loss_total += (loss_bin(out_bin, yb_b) + loss_mul(out_mul, ym_b)).item()
            prob_bin = torch.softmax(out_bin, dim=1).cpu().numpy()
            prob_mul = torch.softmax(out_mul, dim=1).cpu().numpy()
            all_pred_bin.extend(prob_bin.argmax(axis=1))
            all_pred_mul.extend(prob_mul.argmax(axis=1))
            all_prob_bin.append(prob_bin)
            all_prob_mul.append(prob_mul)
            all_true_bin.extend(yb_b.cpu().numpy())
            all_true_mul.extend(ym_b.cpu().numpy())

    best_val_loss = eval_loss_total / len(val_dl)

    pred_bin = np.array(all_pred_bin)
    pred_mul = np.array(all_pred_mul)
    prob_bin = np.vstack(all_prob_bin)
    prob_mul = np.vstack(all_prob_mul)
    true_bin = np.array(all_true_bin)
    true_mul = np.array(all_true_mul)

    mcc_bin = matthews_corrcoef(true_bin, pred_bin)
    mcc_mul = matthews_corrcoef(true_mul, pred_mul)
    mean_mcc = (mcc_bin + mcc_mul) / 2

    cm_bin = confusion_matrix(true_bin, pred_bin)
    cm_mul = confusion_matrix(true_mul, pred_mul)

    metrics_first_output  = print_metrics(true_bin, pred_bin, prob_bin,
                                           f'Trial {FINAL_LABEL} - Binary Task',
                                           class_names=['Benign', 'Malicious'],
                                           verbose=verbose)
    metrics_second_output = print_metrics(true_mul, pred_mul, prob_mul,
                                           f'Trial {FINAL_LABEL} - Multiclass Task',
                                           class_names=label_classes, verbose=verbose)

    # ── Feature importance: reuse from old JSON if reloaded ───────────
    if reloaded and old_feature_importance is not None:
        feature_importance = old_feature_importance
    else:
        feature_importance = None
        try:
            feat_names = list(feature_names) if feature_names is not None \
                         else [f'f{i}' for i in range(n_features)]

            base_acc = (pred_bin == true_bin).mean()
            importances = {}
            X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)

            for i in range(n_features):
                X_perm = X_val_t.clone()
                idx = torch.randperm(X_val_t.shape[0])
                X_perm[:, :, i] = X_val_t[idx, :, i]
                with torch.no_grad():
                    out_b, _ = model(X_perm)
                    p = out_b.argmax(dim=1).cpu().numpy()
                drop = base_acc - (p == true_bin).mean()
                importances[feat_names[i]] = float(drop)

            max_imp = max(importances.values()) or 1.0
            feature_importance = {k: max(v, 0) / max_imp
                                   for k, v in importances.items()}
        except Exception as e:
            print(f"  ⚠️ Feature importance failed (trial {FINAL_LABEL}): {e}")

    # ── Plot confusion matrix ─────────────────────────────────────────
    if verbose != 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                    xticklabels=['Benign', 'Malicious'],
                    yticklabels=['Benign', 'Malicious'])
        axes[0].set_title(f'Trial {FINAL_LABEL} — Binary CM')
        axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
        sns.heatmap(cm_mul, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                    xticklabels=label_classes[:cm_mul.shape[1]],
                    yticklabels=label_classes[:cm_mul.shape[0]])
        axes[1].set_title(f'Trial {FINAL_LABEL} — Multiclass CM')
        axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(f'{PATH_IMG}/cnn_lstm_trial{FINAL_LABEL}_cm.png',
                    dpi=150, bbox_inches='tight')
        plt.show()

        # Loss curve (disponibile solo se addestrato da zero)
        if val_losses is not None:
            best_epoch = int(np.argmin(val_losses))
            fig_l, ax_l = plt.subplots(figsize=(8, 4))
            ax_l.plot(val_losses, marker='o', markersize=3, linewidth=1.5, color='steelblue')
            ax_l.scatter([best_epoch], [val_losses[best_epoch]], color='red', zorder=5,
                         label=f'Best: epoch {best_epoch}, loss={val_losses[best_epoch]:.4f}')
            ax_l.set_xlabel('Epoch'); ax_l.set_ylabel('Val Loss')
            ax_l.set_title(f'Trial {FINAL_LABEL} — Validation Loss Curve')
            ax_l.legend(); ax_l.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f'{PATH_IMG}/cnn_lstm_trial{FINAL_LABEL}_loss.png',
                        dpi=150, bbox_inches='tight')
            plt.show()

    # ── Save (or re-save) artifacts JSON ──────────────────────────────
    object_to_store = {
        'model':              model_path,
        'model_type':         'CNN-LSTM',
        'label':              str(FINAL_LABEL),
        'mcc_bin':            mcc_bin,
        'mcc_mul':            mcc_mul,
        'mean_mcc':           mean_mcc,
        'val_loss':           best_val_loss,
        'cm_bin':             cm_bin.tolist(),
        'cm_mul':             cm_mul.tolist(),
        'feature_importance': feature_importance,
        'params': trial.params if trial is not None else {
            'nb_filters': nb_filters, 'kernel_size': kernel_size,
            'lstm_units_1': lstm_units_1, 'lstm_units_2': lstm_units_2,
            'dropout': dropout, 'lr': lr, 'batch_size': batch_size,
        },
        'metrics_binary':     metrics_first_output,
        'metrics_multiclass': metrics_second_output,
        'correlation_matrix': corr_matrix,
    }

    with open(artifacts_path, 'w') as f:
        json.dump(object_to_store, f, indent=4, default=json_serializer)

    f1_bin = metrics_first_output['f1']
    f1_mul = metrics_second_output['f1']
    f1_mean = (f1_bin + f1_mul) / 2

    print(f"Trial {FINAL_LABEL}: MCC_bin={mcc_bin:.4f}, MCC_mul={mcc_mul:.4f}, mean={mean_mcc:.4f}")
    return f1_mean

## Step 2: Load best models parameters from tuning

### Helper functions

In [42]:
def reload_all_trial_metadata(models_dir='models'):
    """
    Ripopola tabnet_trial_artifacts e cnn_lstm_trial_artifacts
    con soli metadati (no model weights in memoria).
    """
    import re

    for fname in sorted(os.listdir(models_dir)):
        if not fname.endswith('_artifacts.json'):
            continue

        match = re.match(r'(tabnet|cnn_lstm)_trial_(.+)_artifacts\.json', fname)
        if not match:
            continue

        model_type = match.group(1)
        label_str  = match.group(2)
        try:
            label = int(label_str)
        except ValueError:
            label = label_str

        with open(os.path.join(models_dir, fname)) as f:
            art = json.load(f)

        # cm: list → np.ndarray
        if art.get('cm_bin') is not None:
            art['cm_bin'] = np.array(art['cm_bin'])
        if art.get('cm_mul') is not None:
            art['cm_mul'] = np.array(art['cm_mul'])

        art['_live_model'] = None  # esplicito: modello non caricato

        if model_type == 'tabnet':
            tabnet_trial_artifacts[label] = art
        elif model_type == 'cnn_lstm':
            cnn_lstm_trial_artifacts[label] = art

    #print(f"✅  Metadata reload completo:")
    #print(f"    tabnet_trial_artifacts   → {len(tabnet_trial_artifacts)} trials")
    #print(f"    cnn_lstm_trial_artifacts → {len(cnn_lstm_trial_artifacts)} trials")

## Step 3: Experimet with pucktrick 

Once we have a baline we can start evaluate some results by tryng to insert some noise inside the dataset using the pucktrick library.

### Helper function:

Functions to load the whole dataset: use parameters to modify specified column in specified percentage:

In [43]:
def prepare_whole_dataset_from_scratch(column_to_insert_noise, percentage, noise_type):
    global CAT_IDXS, CAT_DIMS, label_classes

    pct_label = f"{PERCENTAGE_TO_USE*100:.1f}pct".replace('.', '_')
    #print(f"📦  Loading dataset and sampling {pct_label} stratified ...")

    # ── 1. Read important features from CSV ───────────────────────────
    imp_df = pd.read_csv('models/important_features.csv')
    important_features = imp_df['feature'].tolist()
    #print(f"📋  Important features from CSV ({len(important_features)}): {important_features}")

    KEEP_ALWAYS = {'Label', 'label_generic', 'Timestamp'}

    # ── 2. Load parquet & keep only important columns ─────────────────
    sdf_full = spark.read.parquet(f'{PATH}/all_elaborated.parquet')
    all_cols = set(sdf_full.columns)
    cols_to_keep = [c for c in sdf_full.columns
                    if c in KEEP_ALWAYS or c in important_features]
    sdf_full = sdf_full.select(*cols_to_keep)
    #print(f"✅  Kept {len(cols_to_keep)} columns (from {len(all_cols)})")

    # ── 3. Classify features (only among those actually kept) ─────────
    FEATURE_COLS = [c for c in important_features if c in set(cols_to_keep)]
    CAT_COLS  = [c for c in CATEGORICAL_FEATURES if c in FEATURE_COLS]
    BIN_COLS  = [c for c in BINARY_FEATURES      if c in FEATURE_COLS]
    CONT_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS and c not in BIN_COLS]

    # ── 4. Cast types ─────────────────────────────────────────────────
    ts_dtype = dict(sdf_full.dtypes).get('Timestamp', 'string')
    if ts_dtype == 'string':
        sdf_full = sdf_full.withColumn(
            'Timestamp',
            F.to_timestamp(F.col('Timestamp'), 'dd/MM/yyyy HH:mm:ss')
        )
        #print("📅  Timestamp string → TimestampType")

    dtypes_map = dict(sdf_full.dtypes)
    for c in FEATURE_COLS:
        if dtypes_map.get(c) not in ('double', 'float'):
            sdf_full = sdf_full.withColumn(c, F.col(c).cast('double'))
    #print(f"✅  Cast {len(FEATURE_COLS)} feature columns → double")

    # ── 5. Remove corrupted 1970 rows ─────────────────────────────────
    n_before = sdf_full.count()
    sdf_full = sdf_full.filter(F.year(F.col('Timestamp')) > 1970)
    n_dropped = n_before - sdf_full.count()
    #print(f"🗑️  Removed {n_dropped:,} rows with year 1970" if n_dropped
    #      else "✅  No 1970 rows found")

    # ── 6. take only 10% of dataset ───────────────────────────────────
    if PERCENTAGE_TO_USE < 1.0:
        fractions = {
            row['label_generic']: PERCENTAGE_TO_USE
            for row in sdf_full.select('label_generic').distinct().collect()
        }
        sdf_sampled = sdf_full.sampleBy('label_generic', fractions=fractions,
                                        seed=RANDOM_SEED)
        #print(f"📦  Stratified {PERCENTAGE_TO_USE*100:.1f}% → {sdf_sampled.count():,} rows")
    else:
        sdf_sampled = sdf_full
        #print(f"📦  Full dataset → {sdf_sampled.count():,} rows")

    # ── 7. Sort by Timestamp (clean) ──────────────────────────────────
    sdf_sampled = sdf_sampled.orderBy('Timestamp')

    # ── 8. Temporal split (Spark) ─────────────────────────────────────
    from pyspark.sql.window import Window
    w_all = Window.orderBy('Timestamp')
    sdf_sampled = sdf_sampled.withColumn('_row_id', F.row_number().over(w_all) - 1)
    sdf_sampled = sdf_sampled.withColumn('_group', (F.col('_row_id') % 3).cast('int'))

    train_clean = sdf_sampled.filter(F.col('_group') < 2).drop('_row_id', '_group')
    temp_clean  = sdf_sampled.filter(F.col('_group') == 2).drop('_row_id', '_group')

    w_temp = Window.orderBy('Timestamp')
    temp_clean = temp_clean.withColumn('_temp_id', F.row_number().over(w_temp) - 1)
    val_clean  = temp_clean.filter((F.col('_temp_id') % 2) == 0).drop('_temp_id')
    test_clean = temp_clean.filter((F.col('_temp_id') % 2) == 1).drop('_temp_id')

    # ── 9. Fit label encoder on CLEAN full dataset ─────────────────────
    indexer = StringIndexer(inputCol='Label', outputCol='Label_enc', handleInvalid='keep')
    indexer_model = indexer.fit(sdf_sampled.drop('_row_id', '_group'))
    label_classes = list(indexer_model.labels)

    def apply_label_encoding(sdf):
        sdf = sdf.withColumn('label_generic_enc', col('label_generic').cast('int'))
        sdf = indexer_model.transform(sdf)
        sdf = sdf.withColumn('Label_enc', col('Label_enc').cast('int'))
        return sdf

    # ── 10. Dirty ONLY the train-set with PuckTrick ───────────────────
    def make_strategy(noise_type: str, affected, percentage: float) -> dict:
        base = {
            "selection_criteria": "all",
            "percentage": percentage,
            "mode": "new",
            "perturbate_data": {
                "distribution": "random",
                "param": {}
            }
        }
        if noise_type == "duplicated":
            # duplicated agisce sulle righe, non richiede affected_features
            return base
        elif noise_type == "labels":
            return {**base, "affected_features": affected}
        else:
            # missing, noise, outliers
            return {
                **base,
                "affected_features": [affected],
                "perturbate_data": {
                    "distribution": "random",
                    "value": [None],
                    "param": {}
                }
            }

    strategy = make_strategy(noise_type, column_to_insert_noise, percentage)
    OBJ = PuckTrick(dataframe=train_clean, engine=Engine.SPARK)

    dirty_train = None
    if noise_type == "duplicated":
        _, dirty_train = OBJ.duplicated(OBJ.original, strategy=strategy)
    elif noise_type == "labels":
        _, dirty_train = OBJ.labels(OBJ.original, strategy=strategy)
    elif noise_type == "missing":
        _, dirty_train = OBJ.missing(OBJ.original, strategy=strategy)
    elif noise_type == "outliers":
        _, dirty_train = OBJ.outlier(OBJ.original, strategy=strategy)
    elif noise_type == "noise":
        _, dirty_train = OBJ.noise(OBJ.original, strategy=strategy)
    else:
        dirty_train = train_clean

    # PuckTrick aggiunge automaticamente '_pucktrick_row_id' → rimuoverla
    dirty_train = dirty_train.drop('_pucktrick_row_id')

    dirty_train = dirty_train.orderBy('Timestamp')
    val_clean   = val_clean.orderBy('Timestamp')
    test_clean  = test_clean.orderBy('Timestamp')
    print(f"sporcato TRAIN con pucktrick: {noise_type} su {column_to_insert_noise} al {percentage*100:.1f}%")

    # ── 11. Encode labels (train/val/test) ────────────────────────────
    dirty_train = apply_label_encoding(dirty_train)
    val_clean   = apply_label_encoding(val_clean)
    test_clean  = apply_label_encoding(test_clean)

    # ── 12. Drop Timestamp AFTER dirty+order ──────────────────────────
    dirty_train = dirty_train.drop('Timestamp')
    val_clean   = val_clean.drop('Timestamp')
    test_clean  = test_clean.drop('Timestamp')

    # ── 13. Build categorical encoders on CLEAN full dataset ──────────
    pdf_full_clean, cat_encoders, cat_dims_dict = preprocess_to_pandas(
        sdf_sampled.drop('_row_id', '_group'), CONT_COLS, CAT_COLS, BIN_COLS
    )

    def preprocess_with_encoders(sdf, continuous_features, categorical_features, binary_features, encoders):
        pdf = sdf.toPandas()
        available = set(pdf.columns)
        cont_cols = [c for c in continuous_features if c in available]
        cat_cols  = [c for c in categorical_features if c in available]
        bin_cols  = [c for c in binary_features if c in available]

        for c in cont_cols:
            pdf[c] = pd.to_numeric(pdf[c], errors='coerce')
        pdf[cont_cols] = pdf[cont_cols].fillna(0.0)

        for c in cat_cols:
            le = encoders.get(c)
            if le is None:
                # fallback (should not happen)
                le = LabelEncoder()
                pdf[c] = le.fit_transform(pdf[c].astype(str))
            else:
                try:
                    pdf[c] = le.transform(pdf[c].astype(str))
                except ValueError:
                    mapping = {cls: i for i, cls in enumerate(le.classes_)}
                    pdf[c] = pdf[c].astype(str).map(mapping).fillna(0).astype(int)

        for c in bin_cols:
            pdf[c] = pd.to_numeric(pdf[c], errors='coerce').fillna(0).astype(int)
        return pdf

    # ── 14. Spark → Pandas (train/val/test) ───────────────────────────
    pdf_train = preprocess_with_encoders(dirty_train, CONT_COLS, CAT_COLS, BIN_COLS, cat_encoders)
    pdf_val   = preprocess_with_encoders(val_clean,   CONT_COLS, CAT_COLS, BIN_COLS, cat_encoders)
    pdf_test  = preprocess_with_encoders(test_clean,  CONT_COLS, CAT_COLS, BIN_COLS, cat_encoders)

    # ── 15. Set TabNet globals ─────────────────────────────────────────
    CAT_IDXS = [FEATURE_COLS.index(c) for c in CAT_COLS]
    CAT_DIMS = [cat_dims_dict[c] for c in CAT_COLS]
    #print(f"\n🧮  Features: {len(FEATURE_COLS)} | Cat: {CAT_COLS} | Bin: {BIN_COLS}")

    # ── 16. Extract arrays & clean ────────────────────────────────────
    X_train_2d = pdf_train[FEATURE_COLS].values.astype(np.float32)
    X_val_2d   = pdf_val[FEATURE_COLS].values.astype(np.float32)
    X_test_2d  = pdf_test[FEATURE_COLS].values.astype(np.float32)

    X_train_2d = np.nan_to_num(X_train_2d, nan=0.0, posinf=0.0, neginf=0.0)
    X_val_2d   = np.nan_to_num(X_val_2d,   nan=0.0, posinf=0.0, neginf=0.0)
    X_test_2d  = np.nan_to_num(X_test_2d,  nan=0.0, posinf=0.0, neginf=0.0)

    X_train_2d = np.clip(X_train_2d, -np.finfo(np.float32).max, np.finfo(np.float32).max)
    X_val_2d   = np.clip(X_val_2d,   -np.finfo(np.float32).max, np.finfo(np.float32).max)
    X_test_2d  = np.clip(X_test_2d,  -np.finfo(np.float32).max, np.finfo(np.float32).max)

    y_tr_bin_2d  = pdf_train['label_generic_enc'].values.astype(int)
    y_tr_mul_2d  = pdf_train['Label_enc'].values.astype(int)
    y_val_bin_2d = pdf_val['label_generic_enc'].values.astype(int)
    y_val_mul_2d = pdf_val['Label_enc'].values.astype(int)
    y_te_bin_2d  = pdf_test['label_generic_enc'].values.astype(int)
    y_te_mul_2d  = pdf_test['Label_enc'].values.astype(int)

    # ── 17. Filter rare multiclass labels (train/val/test) ─────────────
    classes, counts = np.unique(y_tr_mul_2d, return_counts=True)
    rare = classes[counts < MIN_SAMPLES_PER_CLASS]
    if len(rare) > 0:
        rare_labels = [label_classes[c] for c in rare if c < len(label_classes)]
        print(f"⚠️  Dropping {len(rare)} rare classes (< {MIN_SAMPLES_PER_CLASS} samples): {rare_labels}")
        keep_tr = ~np.isin(y_tr_mul_2d, rare)
        keep_va = ~np.isin(y_val_mul_2d, rare)
        keep_te = ~np.isin(y_te_mul_2d, rare)
        X_train_2d, y_tr_bin_2d, y_tr_mul_2d = X_train_2d[keep_tr], y_tr_bin_2d[keep_tr], y_tr_mul_2d[keep_tr]
        X_val_2d,   y_val_bin_2d, y_val_mul_2d = X_val_2d[keep_va], y_val_bin_2d[keep_va], y_val_mul_2d[keep_va]
        X_test_2d,  y_te_bin_2d,  y_te_mul_2d  = X_test_2d[keep_te], y_te_bin_2d[keep_te], y_te_mul_2d[keep_te]

    # ── 17b. Keep only classes present in training (val/test) ─────────
    #   After PuckTrick label perturbation some classes may disappear
    #   from train but still exist in val/test → would cause IndexError.
    train_classes = np.unique(y_tr_mul_2d)
    keep_va_cls = np.isin(y_val_mul_2d, train_classes)
    keep_te_cls = np.isin(y_te_mul_2d, train_classes)
    if not keep_va_cls.all():
        X_val_2d, y_val_bin_2d, y_val_mul_2d = X_val_2d[keep_va_cls], y_val_bin_2d[keep_va_cls], y_val_mul_2d[keep_va_cls]
    if not keep_te_cls.all():
        X_test_2d, y_te_bin_2d, y_te_mul_2d = X_test_2d[keep_te_cls], y_te_bin_2d[keep_te_cls], y_te_mul_2d[keep_te_cls]

    # ── 17c. Remap multiclass labels to contiguous 0..n-1 ────────────
    #   CrossEntropyLoss requires targets in [0, n_classes-1].
    #   After filtering, label indices may have gaps (e.g. [0,1,3,7,14])
    #   → remap to contiguous integers so the model output size matches.
    sorted_classes = sorted(train_classes)
    remap_mul = {int(old): new for new, old in enumerate(sorted_classes)}
    y_tr_mul_2d  = np.vectorize(remap_mul.get)(y_tr_mul_2d).astype(int)
    y_val_mul_2d = np.vectorize(remap_mul.get)(y_val_mul_2d).astype(int)
    y_te_mul_2d  = np.vectorize(remap_mul.get)(y_te_mul_2d).astype(int)
    label_classes = [label_classes[c] for c in sorted_classes if c < len(label_classes)]
    print(f"📋  Remapped {len(sorted_classes)} multiclass labels to 0..{len(sorted_classes)-1}")

    # ── 18. Scale continuous features (fit on train only) ─────────────
    cont_idxs = [FEATURE_COLS.index(c) for c in CONT_COLS]
    if cont_idxs:
        scaler = StandardScaler()
        scaler.fit(X_train_2d[:, cont_idxs])
        X_train_2d[:, cont_idxs] = scaler.transform(X_train_2d[:, cont_idxs])
        X_val_2d[:,   cont_idxs] = scaler.transform(X_val_2d[:,   cont_idxs])
        X_test_2d[:,  cont_idxs] = scaler.transform(X_test_2d[:,  cont_idxs])

    # ── 19. 3D sliding windows for CNN-LSTM ───────────────────────────
    def build_windows_from_arrays(X, yb, ym):
        Xw, ybw, ymw = [], [], []
        for s in range(0, len(X) - WINDOW_SIZE + 1, STEP_SIZE):
            Xw.append(X[s:s + WINDOW_SIZE])
            ybw.append(yb[s + WINDOW_SIZE - 1])
            ymw.append(ym[s + WINDOW_SIZE - 1])
        return np.array(Xw, dtype=np.float32), np.array(ybw), np.array(ymw)

    X_train_3d, y_tr_bin_3d, y_tr_mul_3d = build_windows_from_arrays(X_train_2d, y_tr_bin_2d, y_tr_mul_2d)
    X_val_3d,   y_val_bin_3d, y_val_mul_3d = build_windows_from_arrays(X_val_2d, y_val_bin_2d, y_val_mul_2d)

    print(f"\n📐  TabNet  → train {X_train_2d.shape}, val {X_val_2d.shape}")
    print(f"📐  CNN-LSTM → train {X_train_3d.shape}, val {X_val_3d.shape}")

    # ── 20. Correlation matrix on dirty train (features + label_generic) ──
    corr_cols = FEATURE_COLS + ['label_generic_enc']
    corr_matrix = pdf_train[[c for c in corr_cols if c in pdf_train.columns]].corr().round(4).to_dict()
    print(f"📊  Correlation matrix computed ({len(corr_cols)} cols)")

    return (
        FEATURE_COLS,
        X_train_2d, X_val_2d,
        y_tr_bin_2d, y_tr_mul_2d,
        y_val_bin_2d, y_val_mul_2d,
        X_train_3d, X_val_3d,
        y_tr_bin_3d, y_tr_mul_3d,
        y_val_bin_3d, y_val_mul_3d,
        corr_matrix,
    )

In [44]:
import gc
import ctypes

def clear_memory():
    """Libera RAM: Python heap + GPU + Spark cache + forza glibc malloc_trim."""
    # 1. Python garbage collector
    gc.collect()
    
    # 2. PyTorch GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # 3. Spark: svuota cache e broadcast
    try:
        spark.catalog.clearCache()
    except Exception:
        pass
    
    # 4. Forza il rilascio della memoria al SO (Linux glibc)
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

In [45]:
def experiment_already_exists(tag):
    """Return True if both TabNet and CNN-LSTM artifact JSONs exist for this experiment."""
    tabnet_path   = f"experiments/tabnet_trial_{tag}_artifacts.json"
    cnn_lstm_path = f"experiments/cnn_lstm_trial_{tag}_artifacts.json"
    return os.path.exists(tabnet_path) and os.path.exists(cnn_lstm_path)

### Real experiment

- Timestamp --> tipo data
- FIN Flag Cnt --> binary feature
- Down/Up Ratio --> continuos
- Protocol --> categorica

In [46]:
def run_single_experiment(colonna_da_sporcare, metodo, pct):
    global tabnet_trial_artifacts
    tabnet_trial_artifacts = {}
    global cnn_lstm_trial_artifacts
    cnn_lstm_trial_artifacts = {}
    reload_all_trial_metadata()
    
    ## tabnet params
    best_tabnet_trial_num = 0
    best_tabnet = tabnet_trial_artifacts[best_tabnet_trial_num]["params"]
    
    ## cnn params
    best_cnn_lstm_trial_num = 1
    best_cnn_lstm = cnn_lstm_trial_artifacts[best_cnn_lstm_trial_num]["params"]

    FEATURE_NAMES, X_base_train_2d, X_base_val_2d, y_base_tr_bin_2d, y_base_tr_mul_2d, y_base_val_bin_2d, y_base_val_mul_2d, X_base_train, X_base_val, y_base_tr_bin, y_base_tr_mul, y_base_val_bin, y_base_val_mul, corr_matrix = prepare_whole_dataset_from_scratch(colonna_da_sporcare, pct, metodo)
    new_batch_size = 4096
    
    
    label_model = f'Experiment_{metodo}_{colonna_da_sporcare.replace("/", "_")}_{pct*100:.1f}'
    
    tabnet_multitask_objective(
        X_base_train_2d, X_base_val_2d,
        y_base_tr_bin_2d, y_base_tr_mul_2d,
        y_base_val_bin_2d, y_base_val_mul_2d,
        best_tabnet['N_a'], best_tabnet['N_steps'], best_tabnet['gamma'],
        (best_tabnet['lambda_sparse'] * ((0.001 / PERCENTAGE_TO_USE) ** 0.5)),
        best_tabnet['lr'], 
        new_batch_size,
        best_tabnet['mask_type'],
        verbose=0, trial=None, 
        label_model=label_model ,
        feature_names=FEATURE_NAMES,
        corr_matrix=corr_matrix
    )
    
    cnn_lstm_multitask_objective(   
        X_base_train, X_base_val,
        y_base_tr_bin, y_base_tr_mul,
        y_base_val_bin, y_base_val_mul,
        best_cnn_lstm['nb_filters'], best_cnn_lstm['kernel_size'],
        best_cnn_lstm['lstm_units_1'], best_cnn_lstm['lstm_units_2'],
        best_cnn_lstm['dropout'], best_cnn_lstm['lr']/2, #scaling 
        batch_size=new_batch_size,
        verbose=0, 
        label_model=label_model,
        feature_names=FEATURE_NAMES,
        corr_matrix=corr_matrix
    )

In [47]:
FEATURESS_FOR_EXPERTS = pd.read_csv('models/important_features.csv')['feature'].tolist()
FEATURESS_FOR_EXPERTS.append('Timestamp')
PUCKTRICK_METHODS = ['labels', 'missing', 'outliers', 'noise']
PERCENTAGES = [0.05, 0.1, 0.2, 0.35, 0.5, 0.75]

In [48]:
for colonna_da_sporcare in FEATURESS_FOR_EXPERTS:
    for metodo in PUCKTRICK_METHODS:
        for pct in PERCENTAGES:
            
            if experiment_already_exists(f'Experiment_{metodo}_{colonna_da_sporcare.replace("/", "_")}_{pct*100:.1f}'):
                continue
            
            if metodo == 'labels' and (colonna_da_sporcare != 'FIN Flag Cnt' and colonna_da_sporcare != 'Protocol'):
                continue  # non ha senso sporcare usare il metodo "labels" sulla colonna "label_generic" stessa
            
            run_single_experiment(colonna_da_sporcare, metodo, pct)
            clear_memory()           

[2026-04-29 07:30:31] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-04-29 07:30:31] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-04-29 07:30:31] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-04-29 07:30:31] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-04-29 07:30:31] [INFO] Creazione SparkBackend...
[2026-04-29 07:30:31] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-04-29 07:30:31] [DEBUG] SparkSession già esistente.
[2026-04-29 07:30:31] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-04-29 07:30:31] [INFO] SparkBackend pronto.
[2026-04-29 07:30:31] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-04-29 07:30:31] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-04-29 07:30:31]

sporcato TRAIN con pucktrick: missing su Fwd Act Data Pkts al 5.0%
⏳  Converting Spark → Pandas ...
📊  Shape: (1614745, 12)
✅  Preprocessed: 6 continuous | 2 categorical | 1 binary


26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 07:32:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/29 0

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1076497, 9), val (269124, 9)
📐  CNN-LSTM → train (107645, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)

Early stopping occurred at epoch 14 with best_epoch = 10 and best_val_0_accuracy = 0.96058
Successfully saved model at experiments/tabnet_trial_Experiment_missing_Fwd Act Data Pkts_5.0.zip
✅  Saved TabNet model for trial Experiment_missing_Fwd Act Data Pkts_5.0
Trial Experiment_missing_Fwd Act Data Pkts_5.0: MCC_bin=0.8877, MCC_mul=0.8374, mean=0.8626
✅  Saved CNN-LSTM model for trial Experiment_missing_Fwd Act Data Pkts_5.0
Trial Experiment_missing_Fwd Act Data Pkts_5.0: MCC_bin=0.7252, MCC_mul=0.5816, mean=0.6534


[2026-04-29 08:04:57] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-04-29 08:04:57] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-04-29 08:04:57] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-04-29 08:04:57] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-04-29 08:04:57] [INFO] Creazione SparkBackend...
[2026-04-29 08:04:57] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-04-29 08:04:57] [DEBUG] SparkSession già esistente.
[2026-04-29 08:04:57] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-04-29 08:04:57] [INFO] SparkBackend pronto.
[2026-04-29 08:04:57] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-04-29 08:04:57] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-04-29 08:04:57]

...

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1076497, 9), val (269124, 9)
📐  CNN-LSTM → train (107645, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)

Early stopping occurred at epoch 13 with best_epoch = 9 and best_val_0_accuracy = 0.95817
Successfully saved model at experiments/tabnet_trial_Experiment_noise_Timestamp_50.0.zip
✅  Saved TabNet model for trial Experiment_noise_Timestamp_50.0
Trial Experiment_noise_Timestamp_50.0: MCC_bin=0.8779, MCC_mul=0.8295, mean=0.8537
✅  Saved CNN-LSTM model for trial Experiment_noise_Timestamp_50.0
Trial Experiment_noise_Timestamp_50.0: MCC_bin=0.7844, MCC_mul=0.7664, mean=0.7754


[2026-05-04 05:45:57] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-05-04 05:45:57] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-05-04 05:45:57] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-05-04 05:45:57] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-05-04 05:45:57] [INFO] Creazione SparkBackend...
[2026-05-04 05:45:57] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-05-04 05:45:57] [DEBUG] SparkSession già esistente.
[2026-05-04 05:45:57] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-05-04 05:45:57] [INFO] SparkBackend pronto.
[2026-05-04 05:45:57] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-05-04 05:45:57] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-05-04 05:45:57]

sporcato TRAIN con pucktrick: noise su Timestamp al 75.0%
⏳  Converting Spark → Pandas ...
📊  Shape: (1614745, 12)
✅  Preprocessed: 6 continuous | 2 categorical | 1 binary


26/05/04 05:48:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 05:48:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 05:48:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 05:48:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 05:48:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 05:48:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 0

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1076497, 9), val (269124, 9)
📐  CNN-LSTM → train (107645, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)

Early stopping occurred at epoch 16 with best_epoch = 12 and best_val_0_accuracy = 0.95976
Successfully saved model at experiments/tabnet_trial_Experiment_noise_Timestamp_75.0.zip
✅  Saved TabNet model for trial Experiment_noise_Timestamp_75.0
Trial Experiment_noise_Timestamp_75.0: MCC_bin=0.8862, MCC_mul=0.8333, mean=0.8597
✅  Saved CNN-LSTM model for trial Experiment_noise_Timestamp_75.0
Trial Experiment_noise_Timestamp_75.0: MCC_bin=0.7546, MCC_mul=0.7370, mean=0.7458


In [49]:
PUCKTRICK_METHODS = ['duplicated']

In [50]:
for metodo in PUCKTRICK_METHODS:
    for pct in PERCENTAGES:
        
        if experiment_already_exists(f'Experiment_{metodo}_{colonna_da_sporcare.replace("/", "_")}_{pct*100:.1f}'):
            continue
        
        try:
            run_single_experiment(colonna_da_sporcare, metodo, pct)
        except Exception as e:
            print(f"Error occurred during experiment {metodo} with {colonna_da_sporcare} at {pct*100:.1f}%: {e}")
        finally:
            clear_memory()    

[2026-05-04 06:21:59] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-05-04 06:21:59] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-05-04 06:21:59] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-05-04 06:21:59] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-05-04 06:21:59] [INFO] Creazione SparkBackend...
[2026-05-04 06:21:59] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-05-04 06:21:59] [DEBUG] SparkSession già esistente.
[2026-05-04 06:21:59] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-05-04 06:21:59] [INFO] SparkBackend pronto.
[2026-05-04 06:21:59] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-05-04 06:21:59] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-05-04 06:21:59]

sporcato TRAIN con pucktrick: duplicated su Timestamp al 5.0%
⏳  Converting Spark → Pandas ...
📊  Shape: (1614745, 12)
✅  Preprocessed: 6 continuous | 2 categorical | 1 binary


26/05/04 06:23:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 06:23:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 06:23:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 06:23:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 06:23:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 06:23:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 0

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1130321, 9), val (269124, 9)
📐  CNN-LSTM → train (113028, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)
Stop training because you reached max_epochs = 20 with best_epoch = 19 and best_val_0_accuracy = 0.96119
Successfully saved model at experiments/tabnet_trial_Experiment_duplicated_Timestamp_5.0.zip
✅  Saved TabNet model for trial Experiment_duplicated_Timestamp_5.0
Trial Experiment_duplicated_Timestamp_5.0: MCC_bin=0.8916, MCC_mul=0.8382, mean=0.8649
✅  Saved CNN-LSTM model for trial Experiment_duplicated_Timestamp_5.0
Trial Experiment_duplicated_Timestamp_5.0: MCC_bin=0.7359, MCC_mul=0.5999, mean=0.6679


[2026-05-04 06:57:05] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-05-04 06:57:05] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-05-04 06:57:05] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-05-04 06:57:05] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-05-04 06:57:05] [INFO] Creazione SparkBackend...
[2026-05-04 06:57:05] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-05-04 06:57:05] [DEBUG] SparkSession già esistente.
[2026-05-04 06:57:05] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-05-04 06:57:05] [INFO] SparkBackend pronto.
[2026-05-04 06:57:05] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-05-04 06:57:05] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-05-04 06:57:05]

...

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1614745, 9), val (269124, 9)
📐  CNN-LSTM → train (161470, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)

Early stopping occurred at epoch 13 with best_epoch = 9 and best_val_0_accuracy = 0.9621
Successfully saved model at experiments/tabnet_trial_Experiment_duplicated_Timestamp_50.0.zip
✅  Saved TabNet model for trial Experiment_duplicated_Timestamp_50.0
Trial Experiment_duplicated_Timestamp_50.0: MCC_bin=0.8838, MCC_mul=0.8519, mean=0.8678
✅  Saved CNN-LSTM model for trial Experiment_duplicated_Timestamp_50.0
Trial Experiment_duplicated_Timestamp_50.0: MCC_bin=0.7466, MCC_mul=0.5549, mean=0.6507


[2026-05-04 09:35:20] [INFO] Inizializzazione PuckTrick...
INFO:pucktrick:Inizializzazione PuckTrick...
[2026-05-04 09:35:20] [INFO] Backend richiesto: Engine.SPARK
INFO:pucktrick:Backend richiesto: Engine.SPARK
[2026-05-04 09:35:20] [DEBUG] PySpark availability: True
DEBUG:pucktrick:PySpark availability: True
[2026-05-04 09:35:20] [INFO] Forzo backend Spark.
INFO:pucktrick:Forzo backend Spark.
[2026-05-04 09:35:20] [INFO] Creazione SparkBackend...
[2026-05-04 09:35:20] [INFO] Creazione SparkBackend...
INFO:pucktrick.sparkbackend:Creazione SparkBackend...
[2026-05-04 09:35:20] [DEBUG] SparkSession già esistente.
[2026-05-04 09:35:20] [DEBUG] SparkSession già esistente.
DEBUG:pucktrick.spark:SparkSession già esistente.
[2026-05-04 09:35:20] [INFO] SparkBackend pronto.
[2026-05-04 09:35:20] [INFO] SparkBackend pronto.
INFO:pucktrick.sparkbackend:SparkBackend pronto.
[2026-05-04 09:35:20] [INFO] Backend attivo: Engine.SPARK
INFO:pucktrick:Backend attivo: Engine.SPARK
[2026-05-04 09:35:20]

sporcato TRAIN con pucktrick: duplicated su Timestamp al 75.0%
⏳  Converting Spark → Pandas ...
📊  Shape: (1614745, 12)
✅  Preprocessed: 6 continuous | 2 categorical | 1 binary


26/05/04 09:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 09:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 09:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 09:36:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 09:37:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 09:37:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/04 0

📋  Remapped 15 multiclass labels to 0..14

📐  TabNet  → train (1883869, 9), val (269124, 9)
📐  CNN-LSTM → train (188382, 50, 9), val (26908, 50, 9)
📊  Correlation matrix computed (10 cols)
Stop training because you reached max_epochs = 20 with best_epoch = 18 and best_val_0_accuracy = 0.96461
Successfully saved model at experiments/tabnet_trial_Experiment_duplicated_Timestamp_75.0.zip
✅  Saved TabNet model for trial Experiment_duplicated_Timestamp_75.0
Trial Experiment_duplicated_Timestamp_75.0: MCC_bin=0.8931, MCC_mul=0.8608, mean=0.8769
✅  Saved CNN-LSTM model for trial Experiment_duplicated_Timestamp_75.0
Trial Experiment_duplicated_Timestamp_75.0: MCC_bin=0.6036, MCC_mul=0.5098, mean=0.5567
